Inference

In [ ]:
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"

MATH_ADAPTER = "./math_lora"
CODE_ADAPTER = "./final_code_lora"
CREATIVE_ADAPTER = "./final_creative_lora"

print("Base model:")
print(BASE_MODEL)

print("\nAdapters:")
print("Math     :", MATH_ADAPTER)
print("Code     :", CODE_ADAPTER)
print("Creative :", CREATIVE_ADAPTER)

In [ ]:
# ============================================================
# CELL 2: Verify adapter files
# ============================================================

import os

adapter_paths = {
    "math": MATH_ADAPTER,
    "code": CODE_ADAPTER,
    "creative": CREATIVE_ADAPTER
}

for domain, path in adapter_paths.items():

    print("=" * 60)
    print(domain.upper())
    print("Path:", path)

    if os.path.exists(path):

        print("✅ Directory exists")

        files = os.listdir(path)

        print("Files:")
        for file in files:
            print("  -", file)

    else:
        print("❌ Directory NOT FOUND")

In [ ]:
#Install dependencies
!pip install -q -U transformers peft accelerate bitsandbytes sentencepiece

In [ ]:
# Imports


import os
import time
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from peft import PeftModel

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Load tokenizer


print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

print("✅ Tokenizer loaded.")

print("Vocabulary size:", len(tokenizer))

In [ ]:
# Load Qwen3-4B


print("Loading Qwen3-4B in 4-bit mode...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True,
    trust_remote_code=True
)

base_model.eval()

print("✅ Base model loaded.")

if torch.cuda.is_available():
    print(
        "GPU memory allocated:",
        round(
            torch.cuda.memory_allocated() / 1024**3,
            2
        ),
        "GB"
    )

In [1]:
#Attach adapters

print("Loading Math adapter...")

model = PeftModel.from_pretrained(
    base_model,
    MATH_ADAPTER,
    adapter_name="math"
)

print("✅ Math adapter loaded.")




print("Loading Code adapter...")

model.load_adapter(
    CODE_ADAPTER,
    adapter_name="code"
)

print("✅ Code adapter loaded.")



print("\nLoading Creative adapter...")

model.load_adapter(
    CREATIVE_ADAPTER,
    adapter_name="creative"
)

print("✅ Creative adapter loaded.")

Loading Math adapter...


NameError: name 'PeftModel' is not defined

In [ ]:
# Verify adapter registry


print("Available adapters:")

print(
    model.peft_config.keys()
)

In [ ]:
# Manual Math adapter test

model.set_adapter("math")

print(
    "Active adapter:",
    model.active_adapter
)

query = "Solve 3x + 7 = 25 step by step."

messages = [
    {
        "role": "user",
        "content": query
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():

    output = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )

response = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("\nResponse:\n")
print(response)

In [ ]:
# Manual Code adapter test


model.set_adapter("code")

print(
    "Active adapter:",
    model.active_adapter
)

query = "Write a C++ implementation of binary search."

messages = [
    {
        "role": "user",
        "content": query
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():

    output = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

response = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("\nResponse:\n")
print(response)

In [ ]:
# Manual Creative adapter test


model.set_adapter("creative")

print(
    "Active adapter:",
    model.active_adapter
)

query = (
    "Write a short story about a scientist "
    "who receives a message from the future."
)

messages = [
    {
        "role": "user",
        "content": query
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():

    output = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

response = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("\nResponse:\n")
print(response)